# Week 4 — Exercise 1: your line-fitting workspace

Source: `../exercises/week4_intro_to_bayes_exercises.ipynb`.
Lecture reference: `../lectures/week4_intro_to_bayes.ipynb`, starting at **Fitting a line**.

Choose the repository's `.venv/Scripts/python.exe` as the notebook kernel, then run the setup and plotting cells below. The inference tasks are left for you to implement.

## What you are learning

The predicted mean stays $\mu_i = m x_i+b$. A likelihood describes the scatter around that mean. A prior describes your beliefs about the parameters before conditioning on these data. The posterior combines both:

$$\log p(\theta\mid x,y)=\log p(y\mid x,\theta)+\log p(\theta)+\text{constant}.$$

Reproduce the lecture procedure on `data_0.txt`: choose priors, inspect prior predictions, find the maximum a posteriori (MAP) fit, sample the posterior, and inspect predictions. Then repeat on `data_1.txt` with three noise assumptions.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize, stats
import emcee
import corner

# Works when the notebook is launched from MyWork or the repository root.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "lectures/data/linear_fits/data_0.txt").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Launch this notebook within the course repository.")
DATA_DIR = ROOT / "lectures/data/linear_fits"
rng = np.random.default_rng(2026)
print("Python:", sys.executable)
print("Data:", DATA_DIR)


In [ ]:
data = {}
for name in ("data_0", "data_1"):
    x, y, y_err = np.loadtxt(DATA_DIR / f"{name}.txt", unpack=True)
    data[name] = {"x": x, "y": y, "y_err": y_err}
    print(f"{name}: {len(x)} points; reported uncertainties "
          f"{y_err.min():.3f} to {y_err.max():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, (name, d) in zip(axes, data.items()):
    ax.errorbar(d["x"], d["y"], yerr=d["y_err"], fmt="o", capsize=3)
    ax.set(title=name, xlabel="x")
axes[0].set_ylabel("y")
fig.tight_layout()
plt.show()


## 1. Inspect the data

Compare the direction of the trend, the sizes of the error bars, and how much the points scatter around an imagined straight line. The error bars show **reported** uncertainties; they need not account for all the scatter.

**Your observations:**

- `data_0`:
- `data_1`:


## 2. Reproduce the lecture fit on data_0

Begin with the lecture priors $m\sim\mathcal N(1,1^2)$ and $b\sim\mathcal N(0,1.2^2)$, assumed independent. For this dataset the common standard deviation is $\sigma=0.3$.

For independent Gaussian measurements, use

$$\log L=-\frac12\sum_i\left[\frac{(y_i-\mu_i)^2}{\sigma_i^2}+\log(2\pi\sigma_i^2)\right].$$

This formula works for both common and point-specific uncertainties: ensure a common scalar is broadcast over **all** points before summing the normalization. `sigma` is a standard deviation; `sigma**2` is a variance.

Implement the mean, likelihood, prior, and posterior below. The functions are deliberately unfinished; defining them is safe, but calling them will remind you to finish them.

In [ ]:
d = data["data_0"]
x, y, y_err = d["x"], d["y"], d["y_err"]
sigma_common = 0.3
m_prior = stats.norm(loc=1.0, scale=1.0)
b_prior = stats.norm(loc=0.0, scale=1.2)


def model(theta, x):
    """theta = (m, b). Return predicted mean at each x."""
    raise NotImplementedError("TODO: implement the straight-line mean")


def log_likelihood(theta, x, y, sigma):
    """Independent Gaussian log likelihood; sigma can be scalar or array."""
    raise NotImplementedError("TODO: implement the normalized Gaussian log likelihood")


def log_prior(theta):
    """Sum the independent prior log densities for m and b."""
    raise NotImplementedError("TODO: implement the prior")


def log_posterior(theta, x, y, sigma):
    """Combine prior and likelihood; return -np.inf outside prior support."""
    raise NotImplementedError("TODO: implement the posterior")


### Prior predictions, MAP, and posterior samples

1. Draw slope/intercept pairs from the priors and plot their lines. Are they plausible before considering the observations?
2. Use `optimize.minimize` on **minus** the log posterior. Check the optimizer's success flag before using its result.
3. Sample with `emcee.EnsembleSampler`: there are two parameters. The lecture uses 10 walkers and 5000 steps. Start walkers at slightly different positions; inspect traces and mixing before choosing a burn-in discard.
4. Use `corner.corner` to inspect the joint posterior. Plot lines from posterior draws.
5. For posterior predictive **observations**, add noise drawn from the likelihood to each predicted line. A distribution of lines alone describes uncertainty in the mean.

See the corresponding lecture cells for API examples. Do not rely on a fixed step count alone as proof of convergence.

In [ ]:
# TODO: prior predictive plot
# TODO: MAP optimization and success check
# TODO: emcee sampling, trace inspection, burn-in choice, corner plot
# TODO: mean predictions and noisy posterior predictions


## 3. Apply the same procedure to data_1

Keep $\mu_i=mx_i+b$ and compare:

| Case | Standard deviation | Parameters |
|---|---|---|
| A | One fixed, shared value $\sigma$ | $m,b$ |
| B | Reported $\sigma_{y_i}$ at each point | $m,b$ |
| C | $\sigma_{y_i}+f(mx_i+b)^2$ | $m,b,f$, with $f>0$ |

Choose and record the shared uncertainty used in A. Reconsider the slope/intercept priors: a prior centered at positive slope may be poorly matched to plausible lines for the new dataset. Document your choices and inspect prior predictions.

In B, smaller error bars imply more weight in the residual term. Use the full normalization for unequal uncertainties.

In C, follow the exercise literally: add to the **standard deviation**, then square it inside the Gaussian likelihood. This is also how the provided generator makes the data. Introduce a proper prior supported on $f>0$ and use three parameters in the optimizer/sampler. For example, a half-normal or exponential prior is an option; its scale is a choice you must justify.

Because $\sigma_i$ now depends on $m,b,f$, the normalization is essential: it penalizes arbitrarily widening the distribution. Without it, increasing the noise can reduce residual penalties without cost.

In [ ]:
d1 = data["data_1"]
x1, y1, y_err1 = d1["x"], d1["y"], d1["y_err"]

# TODO A: choose a fixed common sigma and priors, then fit/sample.
# TODO B: repeat using y_err1 as the sigma array.
# TODO C: implement theta=(m,b,f), f>0 prior, and the stated noise model.
# Keep separate results for A, B, C so you can compare them.


## 4. Compare and interpret

- How do the inferred slope/intercept and their uncertainties change across A, B, C?
- Do posterior predictive observations reproduce the visible scatter?
- Is there evidence for additional scatter beyond the reported error bars?
- How sensitive are your conclusions to the chosen priors?

**Your conclusions:**
